# Feature Selection Method Comparison

Each section covers one model. Within each section the same train/test split is used to rank features by four methods — SHAP, Gain, LIME, and Information Gain — then evaluates top-k (k = 1…10) features and plots a per-metric comparison.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import copy
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             average_precision_score)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif
from lime.lime_tabular import LimeTabularExplainer

import lightgbm as lgb
import xgboost as xgb
import importlib
import bootstrap, distribution_estimation, plotting, model_wrapper, performance_comparison


In [ ]:
for mod in (bootstrap, distribution_estimation, plotting, model_wrapper, performance_comparison):
    importlib.reload(mod)

from bootstrap import *
from distribution_estimation import *
from plotting import *
from model_wrapper import *
from performance_comparison import *
print("Modules reloaded.")


In [ ]:
seed = 999
os.makedirs(f"figures/golub_{seed}", exist_ok=True)
os.makedirs(f"shap_results/golub_{seed}", exist_ok=True)


img_path = f"figures/golub_{seed}"
shap_path = f"shap_results/golub_{seed}"

### Data

In [ ]:
golub = pd.read_csv('data/golub.csv')
gene_cols = golub.columns[golub.columns.get_loc('cancer') + 1:]

df = golub.copy()
df["Y"] = df["cancer"].replace({"allB": "ALL", "allT": "ALL", "aml": "AML"})
y = df["Y"].map({"ALL": 0, "AML": 1})

X_raw = df[gene_cols.tolist()].copy()
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X_raw), columns=gene_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


### Model Wrappers

In [ ]:
lgb_wrapper = create_model_wrapper(
    'lightgbm',
    params={
        'objective': 'binary', 'metric': 'average_precision',
        'learning_rate': 0.05, 'max_depth': 6, 'num_leaves': 31,
        'min_data_in_leaf': 20, 'feature_fraction': 0.8,
        'bagging_fraction': 0.8, 'bagging_freq': 5,
        'verbose': -1, 'seed': seed, 'num_threads': 1,
    },
    num_boost_round=300,
)

scale_pos_weight = int((y_train == 0).sum() / (y_train == 1).sum())

xgb_wrapper = create_model_wrapper(
    'xgboost',
    params={
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'eta': 0.05,
    'max_depth': 10,          # much safer
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,   # prevent tiny-leaf splits
    'alpha': 0.1,            # L1 regularization for sparsity
    'scale_pos_weight': scale_pos_weight,
    'max_delta_step': 1,
    },
    num_boost_round=150,
)

rf_wrapper = create_model_wrapper(
    'sklearn', model_class=RandomForestClassifier,
    model_params={'n_estimators': 200, 'random_state': seed, 'max_depth': 20, 'min_samples_leaf': 5},
    use_tree_explainer=True,
)

cb_wrapper = create_model_wrapper(
    'catboost',
    params={
        'random_seed': seed, 'learning_rate': 0.05, 'depth': 6,
        'eval_metric': 'Logloss', 'iterations': 300,
    },
)

gb_wrapper = create_model_wrapper(
    'sklearn', model_class=GradientBoostingClassifier,
    model_params={
        'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05,
        'subsample': 0.8, 'max_features': 'sqrt', 'random_state': seed,
        'class_weight': 'balanced',
    },
    use_tree_explainer=True,
)

lr_wrapper = create_model_wrapper(
    'sklearn', model_class=LogisticRegression,
    model_params={'random_state': seed, 'max_iter': 1000, 'C': 1.0, 'solver': 'saga'},
    use_linear_explainer=True, use_tree_explainer=False,
)

MODELS = {
    'LightGBM':          lgb_wrapper,
    'XGBoost':           xgb_wrapper,
    'RandomForest':      rf_wrapper,
    'CatBoost':          cb_wrapper,
    'GradientBoosting':  gb_wrapper,
    'LogisticRegression': lr_wrapper,
}

### Helper Functions

In [ ]:
def _clone_wrapper(wrapper, input_dim=None):
    if isinstance(wrapper, XGBoostWrapper):
        return XGBoostWrapper(params=wrapper.params.copy(), num_boost_round=wrapper.num_boost_round)
    if isinstance(wrapper, LightGBMWrapper):
        return LightGBMWrapper(params=wrapper.params.copy(), num_boost_round=wrapper.num_boost_round)
    if isinstance(wrapper, SklearnWrapper):
        return SklearnWrapper(
            model_class=wrapper.model_class,
            model_params=wrapper.model_params.copy(),
            use_tree_explainer=wrapper.use_tree_explainer,
            use_linear_explainer=wrapper.use_linear_explainer,
        )
    if isinstance(wrapper, CatBoostWrapper):
        return CatBoostWrapper(params=wrapper.params.copy(), num_boost_round=getattr(wrapper, 'num_boost_round', 100))
    return copy.deepcopy(wrapper)


def _predict_proba(wrapper, X):
    if isinstance(wrapper, XGBoostWrapper):
        p = wrapper.model.predict(xgb.DMatrix(X, enable_categorical=True))
        return np.column_stack([1 - p, p]) if p.ndim == 1 else p
    if isinstance(wrapper, LightGBMWrapper):
        p = wrapper.model.predict(X)
        return np.column_stack([1 - p, p]) if p.ndim == 1 else p
    if isinstance(wrapper, (CatBoostWrapper, SklearnWrapper)):
        return wrapper.model.predict_proba(X)
    raise TypeError(f'Unknown wrapper: {type(wrapper)}')


def get_shap_ranking(wrapper, X_tr, y_tr):
    w = _clone_wrapper(wrapper, input_dim=X_tr.shape[1])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        w.fit(X_tr, y_tr)
        task = 'binary'
        sv = w.compute_shap(X_tr, task=task)
    importance = np.abs(sv).sum(axis=tuple(range(sv.ndim - 1)))
    return (
        pd.DataFrame({'feature': X_tr.columns, 'importance': importance})
        .sort_values('importance', ascending=False).reset_index(drop=True)
    )


def get_gain_ranking(wrapper, X_tr, y_tr):
    w = _clone_wrapper(wrapper, input_dim=X_tr.shape[1])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        w.fit(X_tr, y_tr)
    if isinstance(w, LightGBMWrapper):
        imp = w.model.feature_importance(importance_type='gain').astype(float)
    elif isinstance(w, XGBoostWrapper):
        scores = w.model.get_score(importance_type='gain')
        imp = np.array([scores.get(f, 0.0) for f in X_tr.columns], dtype=float)
    elif isinstance(w, CatBoostWrapper):
        imp = np.array(w.model.get_feature_importance(), dtype=float)
    elif isinstance(w, SklearnWrapper) and hasattr(w.model, 'feature_importances_'):
        imp = w.model.feature_importances_.astype(float)
    elif isinstance(w, SklearnWrapper) and hasattr(w.model, 'coef_'):
        imp = np.abs(w.model.coef_).ravel().astype(float)
    else:
        return None
    return (
        pd.DataFrame({'feature': X_tr.columns.tolist(), 'importance': imp})
        .sort_values('importance', ascending=False).reset_index(drop=True)
    )


def get_lime_ranking(wrapper, X_tr, y_tr, n_samples=10, num_features=50):
    w = _clone_wrapper(wrapper, input_dim=X_tr.shape[1])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        w.fit(X_tr, y_tr)
    X_arr = X_tr.values.astype(float)
    feat_names = X_tr.columns.tolist()
    def _pred(arr, _w=w):
        return _predict_proba(_w, pd.DataFrame(arr, columns=feat_names))
    explainer = LimeTabularExplainer(
        X_arr, feature_names=feat_names,
        class_names=['0', '1'], mode='classification', random_state=42,
    )
    idx = np.random.RandomState(42).choice(len(X_arr), min(n_samples, len(X_arr)), replace=False)
    imp = np.zeros(len(feat_names))
    for i in idx:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            exp = explainer.explain_instance(X_arr[i], _pred, num_features=num_features)
        for fi, w_val in exp.local_exp[1]:
            imp[fi] += abs(w_val)
    imp /= len(idx)
    return (
        pd.DataFrame({'feature': feat_names, 'importance': imp})
        .sort_values('importance', ascending=False).reset_index(drop=True)
    )

def evaluate_topk(wrapper, ranking_df, X_tr, X_te, y_tr, y_te, k_values=range(1, 11)):
    results = {'accuracy': [], 'f1': [], 'ap': [], 'auc': []}
    for k in k_values:
        selected = ranking_df['feature'].head(k).tolist()
        selected_set = set(selected)
        feats = [c for c in X_tr.columns if c in selected_set]  # fixed order

        try:
            w = _clone_wrapper(wrapper, input_dim=len(feats))
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                w.fit(X_tr[feats], y_tr)
                proba = _predict_proba(w, X_te[feats])
            p_pos = proba[:, 1] if proba.shape[1] >= 2 else proba.ravel()
            y_pred = (p_pos >= 0.5).astype(int)
            results['accuracy'].append(accuracy_score(y_te, y_pred))
            results['f1'].append(f1_score(y_te, y_pred, zero_division=0))
            results['ap'].append(average_precision_score(y_te, p_pos))
            results['auc'].append(roc_auc_score(y_te, p_pos))
        except Exception:
            for m in results:
                results[m].append(float('nan'))
    return results


def plot_comparison(model_name, all_results, k_values=list(range(1, 11))):
    metric_keys   = ['accuracy', 'f1',       'ap',                'auc'    ]
    metric_titles = ['Accuracy', 'F1 Score', 'Average Precision', 'AUC-ROC']
    method_styles = {
        'SHAP':   dict(color='steelblue',  linestyle='-',  marker='o'),
        'Gain':   dict(color='darkorange', linestyle='--', marker='s'),
        'MDI':    dict(color='darkorange', linestyle='--', marker='s'),
        'Native': dict(color='darkorange', linestyle='--', marker='s'),
        '|Coef|': dict(color='darkorange', linestyle='--', marker='s'),
        'LIME':   dict(color='green',      linestyle='-.', marker='^'),
        'IG':     dict(color='crimson',    linestyle=':',  marker='D'),
        'Robust': dict(color='purple',     linestyle='-',  marker='X'),
    }
    fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharex=True)
    for ax, key, title in zip(axes, metric_keys, metric_titles):
        for method, res in all_results.items():
            if res is None:
                continue
            ax.plot(k_values, res[key], linewidth=2.0, markersize=6,
                    label=method, **method_styles.get(method, {}))
        ax.set_title(title, fontweight='bold', fontsize=13)
        ax.set_xlabel('k (number of features)', fontsize=14)
        ax.set_ylabel(title, fontsize=14)
        ax.set_xticks(k_values)
        ax.tick_params(axis='both', labelsize=11)
        ax.legend(fontsize=10, loc='lower right')
        ax.grid(True, alpha=0.3)
    # fig.suptitle(f'{model_name} — Feature Selection Comparison',
                #  fontsize=15, fontweight='bold')
    plt.tight_layout()
    fig.savefig(f"figures/golub_{seed}/golub_{model_name}_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_comparison_bar(model_name, all_results, k_values=list(range(1, 11)), y_min_mean=0.9):
    """Bar chart: one row (4 panels), bars are mean over k with std error bars."""
    metric_keys   = ['accuracy', 'f1',       'ap',                'auc'    ]
    metric_titles = ['Accuracy', 'F1 Score', 'Average Precision', 'AUC-ROC']
    method_colors = {
        'SHAP':   'steelblue',
        'Gain':   'darkorange',
        'MDI':    'darkorange',
        'Native': 'darkorange',
        '|Coef|': 'darkorange',
        'LIME':   'green',
        'IG':     'crimson',
        'RoSHAP': 'purple',
    }

    methods = [m for m, r in all_results.items() if r is not None]
    colors  = [method_colors.get(m, 'gray') for m in methods]
    x       = np.arange(len(methods))
    bar_w   = 0.6

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    for ax, key, title in zip(axes, metric_keys, metric_titles):
        means = [np.nanmean(all_results[m][key]) for m in methods]
        stds  = [np.nanstd(all_results[m][key], ddof=0) for m in methods]

        bars = ax.bar(
            x, means, width=bar_w, color=colors, edgecolor='white', linewidth=0.8,
            yerr=stds, capsize=4,
            error_kw=dict(elinewidth=1.2, ecolor='black', capthick=1.2),
        )
        ax.set_title(title, fontweight='bold', fontsize=22)
        ax.set_ylabel(f'Mean {title}', fontsize=18)
        ax.set_xticks(x)
        ax.set_xticklabels(methods, fontsize=20, rotation=30, ha='right')
        ax.tick_params(axis='y', labelsize=16)
        ax.grid(axis='y', alpha=0.3)
        ax.set_axisbelow(True)

        lower_vals = [m - s for m, s in zip(means, stds)]
        upper_vals = [m + s for m, s in zip(means, stds)]
        y_lo = min(y_min_mean, min(lower_vals) - 0.01)
        y_hi = max(upper_vals) + 0.02
        ax.set_ylim(y_lo, y_hi)

        for bar, m, s in zip(bars, means, stds):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                m + s + (y_hi - y_lo) * 0.01,
                f'{m:.3f}',
                ha='center', va='bottom', fontsize=16,
            )

    plt.tight_layout()
    fig.savefig(f"figures/golub_{seed}/golub_{model_name}_bar_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()


### Information Gain Ranking (computed once, model-agnostic)

In [ ]:
print('Computing Information Gain scores...')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    ig_scores = mutual_info_classif(X_train, y_train, random_state=42)

ig_ranking = (
    pd.DataFrame({'feature': X_train.columns, 'importance': ig_scores})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)
print('Top-10 IG features:')
ig_ranking.head(10)


---
## LightGBM

### LightGBM — SHAP

In [ ]:
print('Computing SHAP ranking for LightGBM...')
lightgbm_shap = get_shap_ranking(MODELS['LightGBM'], X_train, y_train)
print('Top-10:')
lightgbm_shap.head(10)


### LightGBM — Gain

In [ ]:
print('Computing Gain ranking for LightGBM...')
lightgbm_gain = get_gain_ranking(MODELS['LightGBM'], X_train, y_train)
if lightgbm_gain is not None:
    print('Top-10:')
    display(lightgbm_gain.head(10))
else:
    print('Gain not supported for LightGBM')


### LightGBM — LIME

_Adjust `n_samples` and `num_features` to trade speed for accuracy._

In [ ]:
print('Computing LIME ranking for LightGBM...')
lightgbm_lime = get_lime_ranking(
    MODELS['LightGBM'], X_train, y_train,
    n_samples=10,    # increase for better estimates
    num_features=50, # top features per explanation
)
print('Top-10:')
lightgbm_lime.head(10)

### LightGBM — Robust

In [ ]:
# first50 = lightgbm_shap['feature'].head(50).tolist()

In [ ]:
lgb_boot_unscreen = boot_multi_repeat_inference_keep_feature(
    X=X_train,  # Use preprocessed data with one-hot encoded categoricals
    y=y_train,
    inner_variance="permutation",
    task="binary",
    n_bootstrap=1000,  
    b_model=1, 
    zero_tol=0,
    model_wrapper=lgb_wrapper,  
    n_jobs=6,        
    show_progress=True
)

In [ ]:
filepath = shap_path+"/lgb_golub.parquet"
# pd.concat(lgb_boot_unscreen, ignore_index=True).to_parquet(
#     filepath,
#     index=False,
#     compression="zstd"
# )
# print("Saved!")

df_all = pd.read_parquet(filepath)
lgb_boot_unscreen = [grp for _, grp in df_all.groupby("bootstrap_id", sort=True)]
print(f"Loaded {len(lgb_boot_unscreen)} bootstrap results")

In [ ]:
lgb_feature_kde = estimate_feature_level_mixture_preagg(
    boot_results=lgb_boot_unscreen,
    bandwidth=0.2,
    kernel="gaussian",
    zero_tol=1e-8
)

In [ ]:
tmp = lgb_feature_kde.copy()

tmp["0PM(%)"] = (tmp["pi_zero"] * 100).round(1).astype(str) + "%"
tmp["SNR"] = tmp["median"] / tmp["sd_estimated"]
tmp["stat"] = tmp["p_nonzero"]* tmp["nonzero_median"] * tmp["SNR"]
                                
top = tmp.nlargest(15, "stat").reset_index(drop=True)
top.index += 1
top.index.name = "Rank"
# top[["feature", "stat", "SNR", "nonzero_median", "sd_estimated", "0PM(%)"]].rename(columns={
#     "feature": "Group",
#     "stat": "importance",
#     "nonzero_median": "Median",
#     "sd_estimated": "Std",
# }).round(3)

lightgbm_robust = top[['feature', 'stat']].rename(columns={'stat': 'importance'})
lightgbm_robust.head(10)


### LightGBM — Evaluation & Comparison

In [ ]:
k_values = list(range(1, 16))

print('Evaluating LightGBM — SHAP...')
lightgbm_res_shap = evaluate_topk(MODELS['LightGBM'], lightgbm_shap, X_train, X_test, y_train, y_test, k_values)

print('Evaluating LightGBM — Gain...')
lightgbm_res_gain = evaluate_topk(MODELS['LightGBM'], lightgbm_gain, X_train, X_test, y_train, y_test, k_values) if lightgbm_gain is not None else None

print('Evaluating LightGBM — LIME...')
lightgbm_res_lime = evaluate_topk(MODELS['LightGBM'], lightgbm_lime, X_train, X_test, y_train, y_test, k_values)

print('Evaluating LightGBM — IG...')
lightgbm_res_ig = evaluate_topk(MODELS['LightGBM'], ig_ranking, X_train, X_test, y_train, y_test, k_values)

print('Evaluating LightGBM — Robust Bootstrap...')
lightgbm_res_robust = evaluate_topk(MODELS['LightGBM'], lightgbm_robust, X_train, X_test, y_train, y_test, k_values)

plot_comparison(
    'LightGBM',
    {'SHAP': lightgbm_res_shap, 'Gain': lightgbm_res_gain,
     'LIME': lightgbm_res_lime, 'IG':   lightgbm_res_ig, 'Robust': lightgbm_res_robust},
    k_values,
)


In [ ]:
plot_comparison_bar(
    'LightGBM',
    {'SHAP': lightgbm_res_shap, 'Gain': lightgbm_res_gain,
     'LIME': lightgbm_res_lime, 'IG':   lightgbm_res_ig, 'RoSHAP': lightgbm_res_robust},
    k_values,
)

---
## XGBoost

### XGBoost — SHAP

In [ ]:
print('Computing SHAP ranking for XGBoost...')
xgboost_shap = get_shap_ranking(MODELS['XGBoost'], X_train, y_train)
print('Top-10:')
xgboost_shap.head(10)

### XGBoost — Gain

In [ ]:
print('Computing Gain ranking for XGBoost...')
xgboost_gain = get_gain_ranking(MODELS['XGBoost'], X_train, y_train)
if xgboost_gain is not None:
    print('Top-10:')
    display(xgboost_gain.head(10))
else:
    print('Gain not supported for XGBoost')


### XGBoost — LIME

_Adjust `n_samples` and `num_features` to trade speed for accuracy._

In [ ]:
print('Computing LIME ranking for XGBoost...')
xgboost_lime = get_lime_ranking(
    MODELS['XGBoost'], X_train, y_train,
    n_samples=10,    # increase for better estimates
    num_features=50, # top features per explanation
)
print('Top-10:')
xgboost_lime.head(10)


### XGBoost — Robust

In [ ]:
# first50 = xgboost_shap['feature'].head(50).tolist()

In [ ]:
xgb_boot_unscreen = boot_multi_repeat_inference_keep_feature(
    X=X_train,
    y=y_train,
    inner_variance="permutation",
    task="binary",
    n_bootstrap=100,  
    b_model=1, 
    zero_tol=0,
    model_wrapper=xgb_wrapper,  
    n_jobs=6,        
    show_progress=True
)

In [ ]:
filepath = shap_path+"/xgb_golub.parquet"
pd.concat(xgb_boot_unscreen, ignore_index=True).to_parquet(
    filepath,
    index=False,
    compression="zstd"
)
print("Saved!")

df_all = pd.read_parquet(filepath)
xgb_boot_unscreen = [grp for _, grp in df_all.groupby("bootstrap_id", sort=True)]
print(f"Loaded {len(xgb_boot_unscreen)} bootstrap results")

In [ ]:
xgb_feature_kde = estimate_feature_level_mixture_preagg(
    boot_results=xgb_boot_unscreen,
    bandwidth=0.2,
    kernel="gaussian",
    zero_tol=1e-8
)

In [ ]:
tmp = xgb_feature_kde.copy()

tmp["0PM(%)"] = (tmp["pi_zero"] * 100).round(1).astype(str) + "%"
tmp["SNR"] = tmp["median"] / tmp["sd_estimated"]
tmp["stat"] = tmp["p_nonzero"]* tmp["nonzero_median"] * tmp["SNR"]
                                
top = tmp.nlargest(15, "stat").reset_index(drop=True)
top.index += 1
top.index.name = "Rank"
# top[["feature", "stat", "SNR", "nonzero_median", "sd_estimated", "0PM(%)"]].rename(columns={
#     "feature": "Group",
#     "stat": "importance",
#     "nonzero_median": "Median",
#     "sd_estimated": "Std",
# }).round(3)

xgboost_robust = top[['feature', 'stat']].rename(columns={'stat': 'importance'})
xgboost_robust.head(10)


In [ ]:
top_k = 10
agg_col = "sum_abs_shap"
top_features = xgboost_robust["feature"].astype(str).head(top_k).tolist()

box_rows = []
for b, res in enumerate(xgb_boot_unscreen):
    part = res.loc[:, ["feature", agg_col]].copy()
    part["feature"] = part["feature"].astype(str)
    part = part[part["feature"].isin(top_features)]
    part["bootstrap_id"] = b
    box_rows.append(part)

box_df = pd.concat(box_rows, ignore_index=True)
box_df = box_df.dropna(subset=[agg_col])
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

plot_data = [
    box_df.loc[box_df["feature"] == f, agg_col].dropna().to_numpy()
    for f in top_features
]

all_vals = np.concatenate(plot_data)
x_min = np.nanpercentile(all_vals, 0.1)
x_max = np.nanpercentile(all_vals, 99)
x_grid = np.linspace(x_min, x_max, 500)

spacing = 0.55
scale = 0.42

fig, ax = plt.subplots(figsize=(5, max(4.5, 0.38 * len(top_features))))

ax.set_facecolor("white")
ax.set_axisbelow(True)
ax.grid(axis="x", color="#DDDDDD", linewidth=1.0, zorder=0)

for i in range(len(top_features)):
    feature = top_features[i]
    vals = plot_data[i]
    y_base = (len(top_features) - 1 - i) * spacing

    if len(vals) > 2 and np.std(vals) > 0:
        kde = gaussian_kde(vals)
        dens = kde(x_grid)
        dens = dens / dens.max() * scale
    else:
        dens = np.zeros_like(x_grid)

    ax.fill_between(
        x_grid,
        y_base,
        y_base + dens,
        facecolor="lightgray",
        edgecolor="black",
        linewidth=0.8,
        alpha=1.0,
        zorder=10 + i,
    )

    ax.plot(
        x_grid,
        y_base + dens,
        color="black",
        linewidth=0.8,
        zorder=11 + i,
    )

    ax.hlines(
        y_base,
        x_min,
        x_max,
        color="black",
        linewidth=0.5,
        zorder=12 + i,
    )

ax.set_yticks([(len(top_features) - 1 - i) * spacing for i in range(len(top_features))])
ax.set_yticklabels(top_features, fontsize=13)

ax.set_xlabel(r"Feature Attribution", fontsize=18)
ax.set_ylabel("")

ax.set_xlim(x_min, x_max)
ax.set_ylim(-0.15, (len(top_features) - 1) * spacing + scale + 0.15)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis="x", labelsize=13, length=0)
ax.tick_params(axis="y", labelsize=15, length=0)

plt.tight_layout()
plt.show()

### XGBoost — Evaluation & Comparison

In [ ]:
k_values = list(range(1, 16))

print('Evaluating XGBoost — SHAP...')
xgboost_res_shap = evaluate_topk(MODELS['XGBoost'], xgboost_shap, X_train, X_test, y_train, y_test, k_values)

print('Evaluating XGBoost — Gain...')
xgboost_res_gain = evaluate_topk(MODELS['XGBoost'], xgboost_gain, X_train, X_test, y_train, y_test, k_values) if xgboost_gain is not None else None

print('Evaluating XGBoost — LIME...')
xgboost_res_lime = evaluate_topk(MODELS['XGBoost'], xgboost_lime, X_train, X_test, y_train, y_test, k_values)

print('Evaluating XGBoost — IG...')
xgboost_res_ig = evaluate_topk(MODELS['XGBoost'], ig_ranking, X_train, X_test, y_train, y_test, k_values)

print('Evaluating XGBoost — Robust Bootstrap...')
xgboost_res_robust = evaluate_topk(MODELS['XGBoost'], xgboost_robust, X_train, X_test, y_train, y_test, k_values)

plot_comparison(
    'XGBoost',
    {'SHAP': xgboost_res_shap, 'Gain': xgboost_res_gain,
     'LIME': xgboost_res_lime, 'IG':   xgboost_res_ig, 'Robust': xgboost_res_robust},
    k_values,
)


In [ ]:
plot_comparison_bar(
    'XGBoost', 
    {'SHAP': xgboost_res_shap, 'Gain': xgboost_res_gain, 
     'LIME': xgboost_res_lime, 'IG': xgboost_res_ig, 'RoSHAP': xgboost_res_robust}, k_values
)

---
## RandomForest

### RandomForest — SHAP

In [ ]:
print('Computing SHAP ranking for RandomForest...')
randomforest_shap = get_shap_ranking(MODELS['RandomForest'], X_train, y_train)
print('Top-10:')
randomforest_shap.head(10)


### RandomForest — Gain

In [ ]:
print('Computing Gain ranking for RandomForest...')
randomforest_gain = get_gain_ranking(MODELS['RandomForest'], X_train, y_train)
if randomforest_gain is not None:
    print('Top-10:')
    display(randomforest_gain.head(10))
else:
    print('Gain not supported for RandomForest')


### RandomForest — LIME

_Adjust `n_samples` and `num_features` to trade speed for accuracy._

In [ ]:
print('Computing LIME ranking for RandomForest...')
randomforest_lime = get_lime_ranking(
    MODELS['RandomForest'], X_train, y_train,
    n_samples=10,    # increase for better estimates
    num_features=50, # top features per explanation
)
print('Top-10:')
randomforest_lime.head(10)

### RandomForest — Robust

In [ ]:
# first50 = randomforest_shap['feature'].head(50).tolist()

In [ ]:
rf_boot_unscreen = boot_multi_repeat_inference_keep_feature(
    X=X_train,
    y=y_train,
    inner_variance="permutation",
    task="binary",
    n_bootstrap=1000,  
    b_model=1, 
    zero_tol=0,
    model_wrapper=rf_wrapper,  
    n_jobs=6,        
    show_progress=True
)

In [ ]:
filepath = shap_path+"/xgb_rf.parquet"
# pd.concat(rf_boot_unscreen, ignore_index=True).to_parquet(
#     filepath,
#     index=False,
#     compression="zstd"
# )
# print("Saved!")

df_all = pd.read_parquet(filepath)
rf_boot_unscreen = [grp for _, grp in df_all.groupby("bootstrap_id", sort=True)]
print(f"Loaded {len(rf_boot_unscreen)} bootstrap results")

In [ ]:
rf_feature_kde = estimate_feature_level_mixture_preagg(
    boot_results=rf_boot_unscreen,
    bandwidth=0.2,
    kernel="gaussian",
    zero_tol=1e-8
)

In [ ]:
tmp = rf_feature_kde.copy()

tmp["0PM(%)"] = (tmp["pi_zero"] * 100).round(1).astype(str) + "%"
tmp["SNR"] = tmp["median"] / tmp["sd_estimated"]
tmp["stat"] = tmp["p_nonzero"]* tmp["nonzero_median"] * tmp["SNR"]
                                
top = tmp.nlargest(15, "stat").reset_index(drop=True)
top.index += 1
top.index.name = "Rank"
# top[["feature", "stat", "SNR", "nonzero_median", "sd_estimated", "0PM(%)"]].rename(columns={
#     "feature": "Group",
#     "stat": "importance",
#     "nonzero_median": "Median",
#     "sd_estimated": "Std",
# }).round(3)

rf_robust = top[['feature', 'stat']].rename(columns={'stat': 'importance'})
rf_robust.head(10)


### RandomForest — Evaluation & Comparison

In [ ]:
k_values = list(range(1, 16))

print('Evaluating RandomForest — SHAP...')
randomforest_res_shap = evaluate_topk(MODELS['RandomForest'], randomforest_shap, X_train, X_test, y_train, y_test, k_values)

print('Evaluating RandomForest — MDI...')
randomforest_res_mdi = evaluate_topk(MODELS['RandomForest'], randomforest_gain, X_train, X_test, y_train, y_test, k_values) if randomforest_gain is not None else None

print('Evaluating RandomForest — LIME...')
randomforest_res_lime = evaluate_topk(MODELS['RandomForest'], randomforest_lime, X_train, X_test, y_train, y_test, k_values)

print('Evaluating RandomForest — IG...')
randomforest_res_ig = evaluate_topk(MODELS['RandomForest'], ig_ranking, X_train, X_test, y_train, y_test, k_values)

print('Evaluating RandomForest — Robust Bootstrap...')
randomforest_res_robust = evaluate_topk(MODELS['RandomForest'], rf_robust, X_train, X_test, y_train, y_test, k_values)

plot_comparison(
    'RandomForest',
    {'SHAP': randomforest_res_shap, 'MDI': randomforest_res_mdi,
     'LIME': randomforest_res_lime, 'IG':   randomforest_res_ig, 'Robust': randomforest_res_robust},
    k_values,
)


In [ ]:
plot_comparison_bar(
    'RandomForest',
    {'SHAP': randomforest_res_shap, 'MDI': randomforest_res_mdi,
     'LIME': randomforest_res_lime, 'IG':   randomforest_res_ig, 'RoSHAP': randomforest_res_robust},
    k_values,
)

---
## CatBoost

### CatBoost — SHAP

In [ ]:
print('Computing SHAP ranking for CatBoost...')
catboost_shap = get_shap_ranking(MODELS['CatBoost'], X_train, y_train)
print('Top-10:')
catboost_shap.head(10)

### CatBoost — Gain

In [ ]:
print('Computing Gain ranking for CatBoost...')
catboost_gain = get_gain_ranking(MODELS['CatBoost'], X_train, y_train)
if catboost_gain is not None:
    print('Top-10:')
    display(catboost_gain.head(10))
else:
    print('Gain not supported for CatBoost')


### CatBoost — LIME

_Adjust `n_samples` and `num_features` to trade speed for accuracy._

In [ ]:
print('Computing LIME ranking for CatBoost...')
catboost_lime = get_lime_ranking(
    MODELS['CatBoost'], X_train, y_train,
    n_samples=10,    # increase for better estimates
    num_features=50, # top features per explanation
)
print('Top-10:')
catboost_lime.head(10)


### CatBoost — Robust

In [ ]:
# first50 = catboost_shap['feature'].head(50).tolist()

In [ ]:
cb_boot_unscreen = boot_multi_repeat_inference_keep_feature(
    X=X_train,
    y=y_train,
    inner_variance="permutation",
    task="binary",
    n_bootstrap=1000,  
    b_model=1, 
    zero_tol=0,
    model_wrapper=cb_wrapper,  
    n_jobs=6,        
    show_progress=True
)

In [ ]:
filepath = shap_path+"/xgb_cb.parquet"
# pd.concat(cb_boot_unscreen, ignore_index=True).to_parquet(
#     filepath,
#     index=False,
#     compression="zstd"
# )
# print("Saved!")

df_all = pd.read_parquet(filepath)
cb_boot_unscreen = [grp for _, grp in df_all.groupby("bootstrap_id", sort=True)]
print(f"Loaded {len(cb_boot_unscreen)} bootstrap results")

In [ ]:
cb_feature_kde = estimate_feature_level_mixture_preagg(
    boot_results=cb_boot_unscreen,
    bandwidth=0.2,
    kernel="gaussian",
    zero_tol=1e-8
)

In [ ]:
tmp = cb_feature_kde.copy()

tmp["0PM(%)"] = (tmp["pi_zero"] * 100).round(1).astype(str) + "%"
tmp["SNR"] = tmp["median"] / tmp["sd_estimated"]
tmp["stat"] = tmp["p_nonzero"]* tmp["nonzero_median"] * tmp["SNR"]
                                
top = tmp.nlargest(15, "stat").reset_index(drop=True)
top.index += 1
top.index.name = "Rank"
# top[["feature", "stat", "SNR", "nonzero_median", "sd_estimated", "0PM(%)"]].rename(columns={
#     "feature": "Group",
#     "stat": "importance",
#     "nonzero_median": "Median",
#     "sd_estimated": "Std",
# }).round(3)

catboost_robust = top[['feature', 'stat']].rename(columns={'stat': 'importance'})
catboost_robust.head(10)


### CatBoost — Evaluation & Comparison

In [ ]:
k_values = list(range(1, 16))

print('Evaluating CatBoost — SHAP...')
catboost_res_shap = evaluate_topk(MODELS['CatBoost'], catboost_shap, X_train, X_test, y_train, y_test, k_values)

print('Evaluating CatBoost — Native...')
catboost_res_native = evaluate_topk(MODELS['CatBoost'], catboost_gain, X_train, X_test, y_train, y_test, k_values) if catboost_gain is not None else None

print('Evaluating CatBoost — LIME...')
catboost_res_lime = evaluate_topk(MODELS['CatBoost'], catboost_lime, X_train, X_test, y_train, y_test, k_values)

print('Evaluating CatBoost — IG...')
catboost_res_ig = evaluate_topk(MODELS['CatBoost'], ig_ranking, X_train, X_test, y_train, y_test, k_values)

print('Evaluating CatBoost — Robust Bootstrap...')
catboost_res_robust = evaluate_topk(MODELS['CatBoost'], catboost_robust, X_train, X_test, y_train, y_test, k_values)

plot_comparison(
    'CatBoost',
    {'SHAP': catboost_res_shap, 'Native': catboost_res_native,
     'LIME': catboost_res_lime, 'IG':   catboost_res_ig, 'Robust': catboost_res_robust},
    k_values,
)


In [ ]:
plot_comparison_bar(
    'CatBoost',
    {'SHAP': catboost_res_shap, 'Native': catboost_res_native,
     'LIME': catboost_res_lime, 'IG':   catboost_res_ig, 'RoSHAP': catboost_res_robust},
    k_values,
)

---
## GradientBoosting

### GradientBoosting — SHAP

In [ ]:
print('Computing SHAP ranking for GradientBoosting...')
gradientboosting_shap = get_shap_ranking(MODELS['GradientBoosting'], X_train, y_train)
print('Top-10:')
gradientboosting_shap.head(10)

### GradientBoosting — Gain

In [ ]:
print('Computing Gain ranking for GradientBoosting...')
gradientboosting_gain = get_gain_ranking(MODELS['GradientBoosting'], X_train, y_train)
if gradientboosting_gain is not None:
    print('Top-10:')
    display(gradientboosting_gain.head(10))
else:
    print('Gain not supported for GradientBoosting')


### GradientBoosting — LIME

_Adjust `n_samples` and `num_features` to trade speed for accuracy._

In [ ]:
print('Computing LIME ranking for GradientBoosting...')
gradientboosting_lime = get_lime_ranking(
    MODELS['GradientBoosting'], X_train, y_train,
    n_samples=10,    # increase for better estimates
    num_features=50, # top features per explanation
)
print('Top-10:')
gradientboosting_lime.head(10)

### GradientBoosting — Robust

In [ ]:
# first50 = gradientboosting_shap['feature'].head(50).tolist()

In [ ]:
gb_boot_unscreen = boot_multi_repeat_inference_keep_feature(
    X=X_train,
    y=y_train,
    inner_variance="permutation",
    task="binary",
    n_bootstrap=500,  
    b_model=1, 
    zero_tol=0,
    model_wrapper=gb_wrapper,  
    n_jobs=6,        
    show_progress=True
)

In [ ]:
filepath = shap_path+"/gb_golub.parquet"
# pd.concat(gb_boot_unscreen, ignore_index=True).to_parquet(
#     filepath,
#     index=False,
#     compression="zstd"
# )
# print("Saved!")

df_all = pd.read_parquet(filepath)
gb_boot_unscreen = [grp for _, grp in df_all.groupby("bootstrap_id", sort=True)]
print(f"Loaded {len(gb_boot_unscreen)} bootstrap results")

In [ ]:
gb_feature_kde = estimate_feature_level_mixture_preagg(
    boot_results=gb_boot_unscreen,
    bandwidth=0.2,
    kernel="gaussian",
    zero_tol=1e-8
)

In [ ]:
tmp = gb_feature_kde.copy()

tmp["0PM(%)"] = (tmp["pi_zero"] * 100).round(1).astype(str) + "%"
tmp["SNR"] = tmp["median"] / tmp["sd_estimated"]
tmp["stat"] = tmp["p_nonzero"]* tmp["nonzero_median"] * tmp["SNR"]
                                
top = tmp.nlargest(15, "stat").reset_index(drop=True)
top.index += 1
top.index.name = "Rank"
# top[["feature", "stat", "SNR", "nonzero_median", "sd_estimated", "0PM(%)"]].rename(columns={
#     "feature": "Group",
#     "stat": "importance",
#     "nonzero_median": "Median",
#     "sd_estimated": "Std",
# }).round(3)

gradientboosting_robust = top[['feature', 'stat']].rename(columns={'stat': 'importance'})
gradientboosting_robust.head(10)


### GradientBoosting — Evaluation & Comparison

In [ ]:
k_values = list(range(1, 16))

print('Evaluating GradientBoosting — SHAP...')
gradientboosting_res_shap = evaluate_topk(MODELS['GradientBoosting'], gradientboosting_shap, X_train, X_test, y_train, y_test, k_values)

print('Evaluating GradientBoosting — MDI...')
gradientboosting_res_mdi = evaluate_topk(MODELS['GradientBoosting'], gradientboosting_gain, X_train, X_test, y_train, y_test, k_values) if gradientboosting_gain is not None else None

print('Evaluating GradientBoosting — LIME...')
gradientboosting_res_lime = evaluate_topk(MODELS['GradientBoosting'], gradientboosting_lime, X_train, X_test, y_train, y_test, k_values)

print('Evaluating GradientBoosting — IG...')
gradientboosting_res_ig = evaluate_topk(MODELS['GradientBoosting'], ig_ranking, X_train, X_test, y_train, y_test, k_values)

print('Evaluating GradientBoosting — Robust Bootstrap...')
gradientboosting_res_robust = evaluate_topk(MODELS['GradientBoosting'], gradientboosting_robust, X_train, X_test, y_train, y_test, k_values)

plot_comparison(
    'GradientBoosting',
    {'SHAP': gradientboosting_res_shap, 'MDI': gradientboosting_res_mdi,
     'LIME': gradientboosting_res_lime, 'IG':   gradientboosting_res_ig, 'Robust': gradientboosting_res_robust},
    k_values,
)


In [ ]:
plot_comparison_bar(
    'GradientBoosting',
    {'SHAP': gradientboosting_res_shap, 'MDI': gradientboosting_res_mdi,
     'LIME': gradientboosting_res_lime, 'IG':   gradientboosting_res_ig, 'RoSHAP': gradientboosting_res_robust},
    k_values,
)

---
## LogisticRegression

### LogisticRegression — SHAP

In [ ]:
print('Computing SHAP ranking for LogisticRegression...')
logisticregression_shap = get_shap_ranking(MODELS['LogisticRegression'], X_train, y_train)
print('Top-10:')
logisticregression_shap.head(10)

### LogisticRegression — Gain

In [ ]:
print('Computing Gain ranking for LogisticRegression...')
logisticregression_gain = get_gain_ranking(MODELS['LogisticRegression'], X_train, y_train)
if logisticregression_gain is not None:
    print('Top-10:')
    display(logisticregression_gain.head(10))
else:
    print('Gain not supported for LogisticRegression')


### LogisticRegression — LIME

_Adjust `n_samples` and `num_features` to trade speed for accuracy._

In [ ]:
print('Computing LIME ranking for LogisticRegression...')
logisticregression_lime = get_lime_ranking(
    MODELS['LogisticRegression'], X_train, y_train,
    n_samples=10,    # increase for better estimates
    num_features=50, # top features per explanation
)
print('Top-10:')
logisticregression_lime.head(10)


### LogisticRegression — Robust

In [ ]:
# first50 = logisticregression_shap['feature'].head(50).tolist()

In [ ]:
lr_boot_unscreen = boot_multi_repeat_inference_keep_feature(
    X=X_train,
    y=y_train,
    inner_variance="permutation",
    task="binary",
    n_bootstrap=1000,  
    b_model=1, 
    zero_tol=0,
    model_wrapper=lr_wrapper,  
    n_jobs=6,        
    show_progress=True
)

In [ ]:
filepath = shap_path+"/lr_golub.parquet"
# pd.concat(lr_boot_unscreen, ignore_index=True).to_parquet(
#     filepath,
#     index=False,
#     compression="zstd"
# )
# print("Saved!")

df_all = pd.read_parquet(filepath)
lr_boot_unscreen = [grp for _, grp in df_all.groupby("bootstrap_id", sort=True)]
print(f"Loaded {len(lr_boot_unscreen)} bootstrap results")

In [ ]:
lr_feature_kde = estimate_feature_level_mixture_preagg(
    boot_results=lr_boot_unscreen,
    bandwidth=0.2,
    kernel="gaussian",
    zero_tol=1e-8
)

In [ ]:
tmp = lr_feature_kde.copy()

tmp["0PM(%)"] = (tmp["pi_zero"] * 100).round(1).astype(str) + "%"
tmp["SNR"] = tmp["median"] / tmp["sd_estimated"]
tmp["stat"] = tmp["p_nonzero"]* tmp["nonzero_median"] * tmp["SNR"]
                                
top = tmp.nlargest(15, "stat").reset_index(drop=True)
top.index += 1
top.index.name = "Rank"
# top[["feature", "stat", "SNR", "nonzero_median", "sd_estimated", "0PM(%)"]].rename(columns={
#     "feature": "Group",
#     "stat": "importance",
#     "nonzero_median": "Median",
#     "sd_estimated": "Std",
# }).round(3)

logisticregression_robust = top[['feature', 'stat']].rename(columns={'stat': 'importance'})
logisticregression_robust.head(10)


### LogisticRegression — Evaluation & Comparison

In [ ]:
k_values = list(range(1, 16))

print('Evaluating LogisticRegression — SHAP...')
logisticregression_res_shap = evaluate_topk(MODELS['LogisticRegression'], logisticregression_shap, X_train, X_test, y_train, y_test, k_values)

print('Evaluating LogisticRegression — |Coef|...')
logisticregression_res_coef = evaluate_topk(MODELS['LogisticRegression'], logisticregression_gain, X_train, X_test, y_train, y_test, k_values) if logisticregression_gain is not None else None

print('Evaluating LogisticRegression — LIME...')
logisticregression_res_lime = evaluate_topk(MODELS['LogisticRegression'], logisticregression_lime, X_train, X_test, y_train, y_test, k_values)

print('Evaluating LogisticRegression — IG...')
logisticregression_res_ig = evaluate_topk(MODELS['LogisticRegression'], ig_ranking, X_train, X_test, y_train, y_test, k_values)

print('Evaluating LogisticRegression — Robust Bootstrap...')
logisticregression_res_robust = evaluate_topk(MODELS['LogisticRegression'], logisticregression_robust, X_train, X_test, y_train, y_test, k_values)

plot_comparison(
    'LogisticRegression',
    {'SHAP': logisticregression_res_shap, '|Coef|': logisticregression_res_coef,
     'LIME': logisticregression_res_lime, 'IG':   logisticregression_res_ig, 'Robust': logisticregression_res_robust},
    k_values,
)


In [ ]:
plot_comparison_bar(
    'LogisticRegression',
    {'SHAP': logisticregression_res_shap, '|Coef|': logisticregression_res_coef,
     'LIME': logisticregression_res_lime, 'IG':   logisticregression_res_ig, 'RoSHAP': logisticregression_res_robust},
    k_values,
)